# 3D Digital Preservation of a Traditional Thai Dancer

This notebook contains two clearly separated workflows:

1. **Experimental image-based visual hull** — reconstructs a coarse surface from the silhouettes of the 200 Blender renders. Its quality depends on accurate masks and exact camera calibration.
2. **Verified Blender reference point cloud** — loads and displays a textured point cloud sampled directly from the evaluated Blender character mesh.

The second workflow is the verified final point-cloud result. It is a **reference/ground-truth export from Blender geometry**, not a reconstruction trained from the 200 images. The 200 images belong to the visual-hull experiment.


## Part A — Experimental image-based visual hull

This section tests dense silhouette carving from the Blender turntable renders. It is retained as a computer-vision experiment and comparison baseline. Do not present its output as the verified final character model unless the masks and camera calibration have been validated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 — Imports and configuration

The normalized dancer is two world units tall. Camera distance is estimated from the median mask
height and the configured field of view, so the default generally works without knowing Blender's
world-unit scale. Set the FOV and camera elevation to the actual Blender values for best accuracy.

In [ ]:
# In Colab, uncomment if a package is missing:
# !pip install opencv-python numpy scipy matplotlib plotly --quiet

import glob, os, re, time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.ndimage import binary_erosion, binary_closing, binary_opening, label

IMG_DIR = "drive/MyDrive/dancer_shots_complete"
IMAGE_GLOB = "*.png"
MAX_IMAGES = 200

# Copy these values from Blender when possible.
HORIZONTAL_FOV_DEG = 50.0
CAMERA_ELEVATION_START_DEG = 0.0
CAMERA_ELEVATION_END_DEG = 70.0
AZIMUTH_TURNS = 2.0       # supplied sequence makes about two rotations while rising
START_ANGLE_DEG = 0.0
TURN_DIRECTION = 1.0       # use -1.0 if the reconstructed views rotate backward

# Reconstruction settings. Start at 96/144, then raise to 128/192 for the final export.
GRID_XY = 96
GRID_Z = 144
VIEW_STRIDE = 3            # 200 images -> about 67 carving views
REQUIRED_SUPPORT = 0.94    # lower to 0.90 only if a few masks have small defects
MASK_DILATION_PX = 2
CHUNK_SIZE = 250_000

OUT_PLY = "dancer_visual_hull_v4.ply"

## 2 — Load renders in numeric order

In [ ]:
def natural_key(path):
    nums = re.findall(r"\d+", os.path.basename(path))
    return int(nums[-1]) if nums else 0

files = sorted(glob.glob(os.path.join(IMG_DIR, IMAGE_GLOB)), key=natural_key)[:MAX_IMAGES]
images_rgba = []
valid_files = []
for path in files:
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        print("Skipping unreadable file:", path)
        continue
    if im.ndim == 2:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGRA)
    elif im.shape[2] == 3:
        im = cv2.cvtColor(im, cv2.COLOR_BGR2BGRA)
    images_rgba.append(im)
    valid_files.append(path)

files = valid_files
N = len(images_rgba)
if N < 12:
    raise ValueError(f"Only {N} images were loaded. Check IMG_DIR; at least 12 views are needed.")
shapes = {im.shape[:2] for im in images_rgba}
if len(shapes) != 1:
    raise ValueError(f"All renders must have the same resolution; found {sorted(shapes)}")
H, W = images_rgba[0].shape[:2]
print(f"Loaded {N} renders at {W} x {H}")

## 3 — Build clean foreground masks

Alpha is used whenever it contains transparency. The fallback estimates the background from the
image border in Lab color space, thresholds the distance automatically, and keeps the largest
connected foreground region. For a detailed costume, alpha renders are strongly preferred.

In [ ]:
def largest_component(mask):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return mask
    # Preserve the main subject plus components at least 0.5% as large (e.g. detached ornaments).
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest = areas.max()
    keep = np.where(areas >= max(20, 0.005 * largest))[0] + 1
    return (np.isin(labels, keep).astype(np.uint8) * 255)

def fallback_background_mask(bgr):
    h, w = bgr.shape[:2]
    bw = max(3, int(min(h, w) * 0.025))
    border = np.concatenate([
        bgr[:bw].reshape(-1, 3), bgr[-bw:].reshape(-1, 3),
        bgr[:, :bw].reshape(-1, 3), bgr[:, -bw:].reshape(-1, 3)
    ])
    bg = np.median(cv2.cvtColor(border.reshape(-1, 1, 3).astype(np.uint8),
                               cv2.COLOR_BGR2LAB).reshape(-1, 3), axis=0)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    dist = np.linalg.norm(lab - bg.astype(np.float32), axis=2)
    scaled = np.clip(dist * (255.0 / max(1.0, np.percentile(dist, 99))), 0, 255).astype(np.uint8)
    _, mask = cv2.threshold(scaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask

def grayscale_scene_mask(bgr, chroma_threshold=5):
    # The supplied floor, world, and shadows have R == G == B. The dancer is colored.
    pixels = bgr.astype(np.int16)
    chroma = pixels.max(axis=2) - pixels.min(axis=2)
    return ((chroma > chroma_threshold).astype(np.uint8) * 255)

def make_mask(im):
    alpha = im[:, :, 3]
    has_real_alpha = np.percentile(alpha, 5) < 250 and np.percentile(alpha, 95) > 5
    if has_real_alpha:
        mask = ((alpha > 8).astype(np.uint8) * 255)
    else:
        mask = grayscale_scene_mask(im[:, :, :3])
        raw_fill = (mask > 0).mean()
        if not 0.005 <= raw_fill <= 0.25:
            mask = fallback_background_mask(im[:, :, :3])
    k = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = largest_component(mask)
    if MASK_DILATION_PX > 0:
        kd = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (2 * MASK_DILATION_PX + 1, 2 * MASK_DILATION_PX + 1))
        mask = cv2.dilate(mask, kd)
    return mask, has_real_alpha

masks, alpha_flags = [], []
for im in images_rgba:
    m, used_alpha = make_mask(im)
    masks.append(m)
    alpha_flags.append(used_alpha)

fill = np.array([(m > 0).mean() for m in masks])
print(f"Alpha masks: {sum(alpha_flags)}/{N}")
print(f"Foreground fill: median={np.median(fill):.3f}, range={fill.min():.3f}..{fill.max():.3f}")
if np.any(fill < 0.005) or np.any(fill > 0.25):
    bad = np.flatnonzero((fill < 0.005) | (fill > 0.25))
    raise RuntimeError(f"Implausible masks at views {bad.tolist()}. Do not run carving until the mask preview contains only the dancer.")

In [ ]:
# Mandatory mask QA: the white region should contain only the complete dancer.
sample_idx = np.linspace(0, N - 1, min(12, N), dtype=int)
fig, axes = plt.subplots(3, 4, figsize=(14, 12))
for ax, idx in zip(axes.ravel(), sample_idx):
    rgb = cv2.cvtColor(images_rgba[idx][:, :, :3], cv2.COLOR_BGR2RGB)
    overlay = rgb.copy()
    overlay[masks[idx] == 0] = (overlay[masks[idx] == 0] * 0.15).astype(np.uint8)
    ax.imshow(overlay)
    ax.set_title(f"view {idx} | fill {fill[idx]:.2f}")
    ax.axis("off")
for ax in axes.ravel()[len(sample_idx):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4 — Camera model for the Blender turntable

The camera looks at the origin, Z is up, and its azimuth advances evenly through 360°. Because
turntable geometry is known, there is no incremental pose drift, arbitrary translation scale, or
fragile loop closure.

In [ ]:
def mask_bbox(mask):
    ys, xs = np.nonzero(mask)
    return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1

boxes = np.array([mask_bbox(m) for m in masks], dtype=float)
heights = (boxes[:, 3] - boxes[:, 1]) / H
# The supplied spiral sequence changes framing. Center every clean silhouette separately.
C_XS = (boxes[:, 0] + boxes[:, 2]) / 2.0
C_YS = (boxes[:, 1] + boxes[:, 3]) / 2.0
fov_x = np.radians(HORIZONTAL_FOV_DEG)
f_x = (W / 2.0) / np.tan(fov_x / 2.0)
fov_y = 2.0 * np.arctan((H / W) * np.tan(fov_x / 2.0))
f_y = (H / 2.0) / np.tan(fov_y / 2.0)
K = np.array([[f_x, 0, W / 2.0], [0, f_y, H / 2.0], [0, 0, 1.0]])

SUBJECT_HEIGHT = 2.0
camera_radius = SUBJECT_HEIGHT / (2.0 * np.percentile(heights, 90) * np.tan(fov_y / 2.0))
camera_radius = float(np.clip(camera_radius, 1.8, 8.0))
print(f"Estimated normalized camera distance: {camera_radius:.3f}")
print(f"Per-view silhouette centers enabled; median center=({np.median(C_XS):.1f}, {np.median(C_YS):.1f})")

def look_at_world_to_camera(position, target=np.zeros(3)):
    # OpenCV camera coordinates: +x right, +y down, +z forward.
    forward = target - position
    forward /= np.linalg.norm(forward)
    right = np.cross(forward, np.array([0.0, 0.0, 1.0]))
    right /= np.linalg.norm(right)
    down = np.cross(forward, right)
    R = np.vstack([right, down, forward])
    t = -R @ position
    return R, t

angles = START_ANGLE_DEG + TURN_DIRECTION * np.linspace(0.0, 360.0 * AZIMUTH_TURNS, N, endpoint=False)
elevations = np.linspace(CAMERA_ELEVATION_START_DEG, CAMERA_ELEVATION_END_DEG, N)
camera_poses = []
for angle_deg, elevation_deg in zip(angles, elevations):
    a = np.radians(angle_deg)
    elevation = np.radians(elevation_deg)
    pos = np.array([
        camera_radius * np.sin(a) * np.cos(elevation),
        -camera_radius * np.cos(a) * np.cos(elevation),
        camera_radius * np.sin(elevation),
    ])
    camera_poses.append(look_at_world_to_camera(pos))

## 5 — Carve the dense visual hull

Each voxel is projected into many silhouettes. A voxel is retained when it lies inside almost
all masks. This reconstructs a solid body volume; the final point cloud contains only its surface.

In [ ]:
# Bounds are normalized around a two-unit-tall dancer. Increase XY_EXTENT if hands are clipped.
XY_EXTENT = 0.80
xs = np.linspace(-XY_EXTENT, XY_EXTENT, GRID_XY, dtype=np.float32)
ys = np.linspace(-XY_EXTENT, XY_EXTENT, GRID_XY, dtype=np.float32)
zs = np.linspace(-1.05, 1.05, GRID_Z, dtype=np.float32)
xx, yy, zz = np.meshgrid(xs, ys, zs, indexing="ij")
voxels = np.column_stack([xx.ravel(), yy.ravel(), zz.ravel()])

view_ids = np.arange(0, N, VIEW_STRIDE, dtype=int)
votes = np.zeros(len(voxels), dtype=np.uint16)
valid_views = np.zeros(len(voxels), dtype=np.uint16)
t0 = time.time()

for count, view_idx in enumerate(view_ids, 1):
    R, t = camera_poses[view_idx]
    mask = masks[view_idx]
    for start in range(0, len(voxels), CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, len(voxels))
        pc = voxels[start:stop] @ R.T + t
        zc = pc[:, 2]
        good_z = zc > 1e-6
        u = np.rint(f_x * pc[:, 0] / np.maximum(zc, 1e-6) + C_XS[view_idx]).astype(np.int32)
        v = np.rint(f_y * pc[:, 1] / np.maximum(zc, 1e-6) + C_YS[view_idx]).astype(np.int32)
        inside = good_z & (u >= 0) & (u < W) & (v >= 0) & (v < H)
        valid_views[start:stop] += inside.astype(np.uint16)
        local_vote = np.zeros(stop - start, dtype=bool)
        ids = np.flatnonzero(inside)
        local_vote[ids] = mask[v[ids], u[ids]] > 0
        votes[start:stop] += local_vote.astype(np.uint16)
    if count % 10 == 0 or count == len(view_ids):
        print(f"view {count}/{len(view_ids)} | elapsed {time.time() - t0:.1f}s")

support = votes / np.maximum(valid_views, 1)
occupied_flat = (valid_views == len(view_ids)) & (support >= REQUIRED_SUPPORT)
occupied = occupied_flat.reshape(GRID_XY, GRID_XY, GRID_Z)
if occupied.sum() == 0:
    raise RuntimeError("No occupied voxels. Recheck mask quality, FOV, turn direction, and camera elevation.")

surface = occupied & ~binary_erosion(occupied, structure=np.ones((3, 3, 3), bool))
surface_points = voxels[surface.ravel()]
print(f"Occupied voxels: {occupied.sum():,}")
print(f"Surface points: {len(surface_points):,}")

## 6 — Color the surface and export PLY

Each point is colored from the render whose azimuth most directly faces that side of the model.
If a projected point falls outside that mask, nearby views are tried as fallbacks.

In [ ]:
def project_points(points, pose, view_idx):
    R, t = pose
    pc = points @ R.T + t
    u = np.rint(f_x * pc[:, 0] / pc[:, 2] + C_XS[view_idx]).astype(np.int32)
    v = np.rint(f_y * pc[:, 1] / pc[:, 2] + C_YS[view_idx]).astype(np.int32)
    return u, v, pc[:, 2]

point_angles = (np.degrees(np.arctan2(surface_points[:, 0], -surface_points[:, 1])) -
                START_ANGLE_DEG) * TURN_DIRECTION
angle_step = 360.0 * AZIMUTH_TURNS / N
views_per_turn = max(1, int(round(N / AZIMUTH_TURNS)))
preferred = np.mod(np.rint(point_angles / angle_step).astype(int), views_per_turn)
colors = np.full((len(surface_points), 3), 190, dtype=np.uint8)
unresolved = np.ones(len(surface_points), dtype=bool)

# Try the preferred view, then neighboring views up to roughly +/-18 degrees.
offsets = [0]
for k in range(1, max(2, N // 20) + 1):
    offsets.extend([k, -k])
for offset in offsets:
    if not unresolved.any():
        break
    candidates = np.flatnonzero(unresolved)
    view_for_point = np.mod(preferred[candidates] + offset, N)
    for view_idx in np.unique(view_for_point):
        ids = candidates[view_for_point == view_idx]
        u, v, zc = project_points(surface_points[ids], camera_poses[view_idx], view_idx)
        good = (zc > 0) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
        loc = np.flatnonzero(good)
        if len(loc):
            loc = loc[masks[view_idx][v[loc], u[loc]] > 0]
        if len(loc):
            chosen = ids[loc]
            colors[chosen] = images_rgba[view_idx][v[loc], u[loc], :3][:, ::-1]
            unresolved[chosen] = False
print(f"Colored {len(colors) - unresolved.sum():,}/{len(colors):,} surface points")

def write_ply(path, points, rgb):
    header = ("ply\nformat ascii 1.0\n" + f"element vertex {len(points)}\n" +
              "property float x\nproperty float y\nproperty float z\n" +
              "property uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n")
    with open(path, "w") as f:
        f.write(header)
        for p, c in zip(points, rgb):
            f.write(f"{p[0]:.6f} {p[1]:.6f} {p[2]:.6f} {int(c[0])} {int(c[1])} {int(c[2])}\n")

write_ply(OUT_PLY, surface_points, colors)
print(f"Saved {OUT_PLY}")

## 7 — Quality-control views

The figure should look like a complete person from all four sides. If it is twisted, correct
`TURN_DIRECTION`. If it is uniformly too thick/thin or clipped, correct the Blender FOV and camera
elevation first; then adjust `REQUIRED_SUPPORT` by at most a few percent.

In [ ]:
show = surface_points
show_colors = colors
if len(show) > 80_000:
    rng = np.random.default_rng(7)
    ids = rng.choice(len(show), 80_000, replace=False)
    show, show_colors = show[ids], show_colors[ids]

fig = plt.figure(figsize=(15, 10))
for n, (elev, azim) in enumerate([(5, 0), (5, 90), (5, 180), (5, 270)], 1):
    ax = fig.add_subplot(2, 2, n, projection="3d")
    ax.scatter(show[:, 0], show[:, 1], show[:, 2], c=show_colors / 255.0, s=0.35)
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect(np.ptp(show, axis=0))
    ax.set_title(f"azimuth {azim}°")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig = go.Figure(go.Scatter3d(
    x=show[:, 0], y=show[:, 1], z=show[:, 2], mode="markers",
    marker=dict(size=1.6, color=[f"rgb({r},{g},{b})" for r, g, b in show_colors], opacity=0.9)
))
fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, b=0, t=30),
                  title="Dense dancer visual hull")
fig.show()

## Part B — Verified textured Blender reference point cloud

The Blender exporter samples the evaluated dancer mesh surface after applying the pose and modifiers, then assigns UV-texture colors to the points. The portable ASCII file contains 100,000 points and avoids the binary-PLY compatibility problem encountered in Colab.

Required input file:

`dancer_exact_pointcloud_textured_100k_ascii.ply`


In [ ]:
# Install Open3D only if the current Colab runtime does not already have it.
!pip install -q open3d


In [ ]:
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go

PLY_PATH = Path("dancer_exact_pointcloud_textured_100k_ascii.ply")
HTML_PATH = Path("dancer_pointcloud_interactive.html")

if not PLY_PATH.exists():
    raise FileNotFoundError(
        f"{PLY_PATH} was not found. Upload it to the Colab Files panel, "
        "or change PLY_PATH to its Google Drive location."
    )


In [ ]:
pcd = o3d.io.read_point_cloud(str(PLY_PATH))
points = np.asarray(pcd.points, dtype=np.float64)
colors = np.asarray(pcd.colors, dtype=np.float64)

if len(points) == 0:
    raise RuntimeError("The PLY loaded but contains no points.")
if points.shape != colors.shape or points.shape[1] != 3:
    raise RuntimeError(
        f"Unexpected data shapes: points={points.shape}, colors={colors.shape}"
    )
if not np.isfinite(points).all():
    raise RuntimeError("The PLY contains NaN or infinite coordinates.")

bbox_size = np.ptp(points, axis=0)
if np.any(bbox_size <= 0) or np.any(bbox_size > 100):
    raise RuntimeError(
        f"Implausible bounding box {bbox_size}; verify the PLY format and file."
    )

print(f"Points: {len(points):,}")
print("Bounding-box size:", bbox_size)
print("Has colors:", pcd.has_colors())
print("Unique RGB colors:", len(np.unique((np.clip(colors, 0, 1) * 255).astype(np.uint8), axis=0)))


In [ ]:
# Limit the displayed points only when necessary; the source PLY remains unchanged.
display_points = points
display_colors = colors
MAX_DISPLAY_POINTS = 100_000

if len(display_points) > MAX_DISPLAY_POINTS:
    rng = np.random.default_rng(7)
    ids = rng.choice(len(display_points), MAX_DISPLAY_POINTS, replace=False)
    display_points = display_points[ids]
    display_colors = display_colors[ids]

rgb_values = (np.clip(display_colors, 0, 1) * 255).astype(np.uint8)
rgb_strings = [f"rgb({r},{g},{b})" for r, g, b in rgb_values]

fig = go.Figure(
    go.Scatter3d(
        x=display_points[:, 0],
        y=display_points[:, 1],
        z=display_points[:, 2],
        mode="markers",
        marker=dict(size=1.2, color=rgb_strings, opacity=0.95),
        hovertemplate="x: %{x:.4f}<br>y: %{y:.4f}<br>z: %{z:.4f}<extra></extra>",
    )
)

fig.update_layout(
    title="Textured Thai Dancer Point Cloud",
    scene=dict(
        aspectmode="data",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgb(25,25,25)",
    ),
    paper_bgcolor="rgb(25,25,25)",
    font=dict(color="white"),
    margin=dict(l=0, r=0, b=0, t=45),
    width=900,
    height=900,
)

fig.show()


In [ ]:
# Run this cell after the visualization cell because it uses the existing `fig` object.
fig.write_html(str(HTML_PATH), include_plotlyjs=True)
print(f"Saved: {HTML_PATH.resolve()}")

# This import works in Google Colab. If running locally, the HTML is already saved.
try:
    from google.colab import files
    files.download(str(HTML_PATH))
except ImportError:
    print("Not running in Colab; open the saved HTML file locally.")


## Interpretation and project status

- The textured reference point cloud preserves the dancer's recognizable pose, headdress, hands, costume shape, and surface colors.
- The reference PLY is generated from Blender's known mesh, so it serves as a high-quality preservation artifact and ground truth.
- The visual hull remains an image-based experiment. It should be evaluated separately rather than mixed with the direct mesh export.
- The next proposal-aligned milestone is body-and-hand landmark extraction, normalization, and export to `dance_pose.json` for Blender armature retargeting.


## Troubleshooting order

### Visual-hull experiment

1. Confirm every mask contains the complete dancer and excludes the floor and background.
2. Confirm all renders use consistent resolution, crop, lens/FOV, and known camera poses.
3. Copy the exact Blender FOV, camera path, and elevation values into the configuration.
4. Flip `TURN_DIRECTION` if the reconstruction twists.
5. Increase voxel-grid resolution only after the coarse preview is correct.

### Reference point cloud

1. Use `dancer_exact_pointcloud_textured_100k_ascii.ply` in Colab.
2. Confirm the loader reports 100,000 points and a bounding box close to `[1.275, 0.657, 1.902]`.
3. If the file is missing after a runtime reset, upload it again or use a persistent Google Drive path.
4. Do not use the earlier binary-loader experiments that produced coordinates near `1e38`.
